# Rozdział 6 — Sygnał analityczny, faza i informacja o kierunku

Notebook odtwarza E6.1–E6.12 i generuje osobne figury PNG/SVG do
`manuscript/chapters/ch06_sygnal_analityczny_faza_kierunek.md`.
Źródła: kanoniczny dokument M5 i udostępniony notebook M5 v.3.
Wszystkie dane są syntetyczne; model dopplerowski nie jest zapisem pacjenta
ani metodą oceny klinicznej. Zakres 80–200/min jest zakresem poszukiwania
w modelu dydaktycznym, nie normą ani progiem diagnostycznym.

Zachowaj `src/` i strukturę pakietu. Uruchomienie z katalogu pakietu albo
`notebooks/`, Python 3.12 i biblioteki z `requirements_ch06.txt`.
Wszystkie zwykłe komórki można wykonać przez `python scripts/run_ch06.py`.
Wyjaśnienia konieczne do rozumienia metod znajdują się w tekście rozdziału.
Nie osadzamy obrazów w base64; poniżej są odwołania do osobnych plików.


In [1]:
from pathlib import Path
import sys,json,platform
import numpy as np
import scipy
from scipy import signal
import matplotlib
import matplotlib.pyplot as plt
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent] if (p/'src/bspsbook/hilbert_complex.py').is_file()),None)
if ROOT is None:raise RuntimeError('Uruchom z katalogu pakietu lub notebooks/.')
sys.path.insert(0,str(ROOT/'src'))
from bspsbook.plotting import configure_plots,save_figure
from bspsbook.hilbert_complex import analytic_dft,split_directions,analytic_features,packet,centered_spectrum
from bspsbook.rhythm import lag_bounds,objectives,select_optimum,estimate_track,evaluate_track,reject_fast_changes
from bspsbook.signals import moving_dq,rate_profile
configure_plots(chapter=6);FIG=ROOT/'figures/generated'
CHECKS=[];RESULTS={}
def check(name,condition):
    if not bool(condition):raise AssertionError(name)
    CHECKS.append(name);print('OK:',name)
def rmse(y,r):return float(np.sqrt(np.mean(abs(np.asarray(y)-np.asarray(r))**2)))
ENV={'python':platform.python_version(),'numpy':np.__version__,'scipy':scipy.__version__,
     'matplotlib':matplotlib.__version__,'seeds':[20260719,20260720,20260721],
     'rng':'numpy.random.default_rng / PCG64'}
print(json.dumps(ENV,ensure_ascii=False,indent=2))


{
  "python": "3.12.14",
  "numpy": "2.3.5",
  "scipy": "1.17.0",
  "matplotlib": "3.10.8",
  "seeds": [
    20260719,
    20260720,
    20260721
  ],
  "rng": "numpy.random.default_rng / PCG64"
}


## E6.1 — znak Q, kierunek rotacji i brak informacji w |FFT|; rys. 6.1

fs=1000 Hz, T=0.2 s, f0=65 Hz, a=1 j.u. D=cos, Q=±sin;
brak dodatkowego okna, całkowita liczba 13 okresów. Widma FFT/N bez zero-paddingu.
Konwencja c=D+jQ, znaki + i − są matematyczne; fizycznych etykiet nie nadano.

![Rys. 6.1](../figures/generated/fig_06_01_signed_rotation.png)


In [2]:
fs=1000.;t=np.arange(200)/fs;D=np.cos(2*np.pi*65*t);Q=np.sin(2*np.pi*65*t)
cp=D+1j*Q;cn=D-1j*Q
check('E6.1: zmiana znaku Q zachowuje moduł FFT kanału',np.allclose(abs(np.fft.fft(Q)),abs(np.fft.fft(-Q))))
check('E6.1: części rzeczywiste przeciwnych rotacji identyczne',np.allclose(cp.real,cn.real))
check('E6.1: cosinus jest sumą połówek przeciwnych rotacji',np.allclose(.5*(cp+cn),D))
fig,ax=plt.subplots(2,2,figsize=(9.4,6.2),layout='constrained')
mask=t<.05
ax[0,0].plot(t[mask]*1000,D[mask],label='D');ax[0,0].plot(t[mask]*1000,Q[mask],'--',label='Q');ax[0,0].plot(t[mask]*1000,-Q[mask],':',label='−Q')
ax[0,0].set(xlabel='Czas [ms]',ylabel='Amplituda [j.u.]',title='a) Różna relacja kanałów');ax[0,0].legend()
f,FQ=centered_spectrum(Q,fs);_,FQm=centered_spectrum(-Q,fs)
ax[0,1].plot(f,abs(FQ),label='|FFT(Q)| / N');ax[0,1].plot(f,abs(FQm),'--',label='|FFT(−Q)| / N')
ax[0,1].set(xlim=(-100,100),xlabel='Częstotliwość [Hz]',ylabel='Moduł [j.u.]',title='b) Te same osobne moduły widm');ax[0,1].legend(fontsize=7)
for c,label,ls in [(cp,'c=D+jQ','-'),(cn,'c=D−jQ','--')]:
    ax[1,0].plot(c[:16].real,c[:16].imag,ls,label=label)
    j=3;ax[1,0].annotate('',xy=(c[j+1].real,c[j+1].imag),xytext=(c[j].real,c[j].imag),arrowprops={'arrowstyle':'->','color':'black'})
    f,C=centered_spectrum(c,fs);ax[1,1].plot(f,abs(C),ls,label=label)
ax[1,0].set(xlabel='D [j.u.]',ylabel='Q [j.u.]',title='c) Przeciwne zwroty obiegu',aspect='equal');ax[1,0].legend(fontsize=7)
ax[1,1].set(xlim=(-100,100),xlabel='Częstotliwość [Hz]',ylabel='Moduł [j.u.]',title='d) Znak w widmie zespolonym');ax[1,1].legend(fontsize=7)
RESULTS['E6.1']={'positive_peak_Hz':float(f[np.argmax(abs(np.fft.fftshift(np.fft.fft(cp))))]),'negative_peak_Hz':-65.}
save_figure(fig,FIG,'fig_06_01_signed_rotation')


OK: E6.1: zmiana znaku Q zachowuje moduł FFT kanału
OK: E6.1: części rzeczywiste przeciwnych rotacji identyczne
OK: E6.1: cosinus jest sumą połówek przeciwnych rotacji
Figura: figures/generated/fig_06_01_signed_rotation.png (+ SVG)


## E6.2 — maska sygnału analitycznego; rys. 6.2

Podstawowy ton17 Hz, fs500 Hz, 1 s, a=1 j.u. Kontrole parzystej/nieparzystej
długości, DC i Nyquista, zgodność z scipy.signal.hilbert.
Widmo sygnału analitycznego na nieprzedłużonej siatce DFT: DC/Nyquista
nie zaliczamy do zwykłych ujemnych częstotliwości przy kontroli jednostronności.
Test energii: Ez=2Ex−(|X[0]|²+|X[N/2]|²)/N, składnik Nyquista tylko dla parzystego N.

![Rys. 6.2](../figures/generated/fig_06_02_analytic_signal.png)


In [3]:
fs=500.;t=np.arange(500)/fs;x=np.cos(2*np.pi*17*t)
z,mask=analytic_dft(x)
check('E6.2: Hilbert cos = sin dla całkowitej liczby okresów',np.allclose(z.imag,np.sin(2*np.pi*17*t),atol=1e-12))
errors=[]
for N in [499,500]:
    n=np.arange(N);xx=1.2+np.cos(2*np.pi*17*n/fs)
    if N%2==0:xx+=.3*(-1.)**n
    zz,m=analytic_dft(xx);X=np.fft.fft(xx);Z=np.fft.fft(zz)
    check(f'E6.2: ręczna maska = SciPy, N={N}',np.allclose(zz,signal.hilbert(xx),atol=1e-12))
    check(f'E6.2: Re(z)=x, N={N}',np.allclose(zz.real,xx,atol=1e-12))
    negative=np.fft.fftfreq(N)<0
    if N%2==0:negative[N//2]=False
    check(f'E6.2: zwykła ujemna półoś usunięta, N={N}',np.linalg.norm(Z[negative])<1e-10)
    special=abs(X[0])**2+(abs(X[N//2])**2 if N%2==0 else 0)
    energy_ref=2*np.sum(xx*xx)-special/N
    check(f'E6.2: bilans energii z DC/Nyquistą, N={N}',np.isclose(np.sum(abs(zz)**2),energy_ref,atol=1e-10))
    errors.append(float(np.max(abs(zz-signal.hilbert(xx)))))
fig,ax=plt.subplots(2,1,figsize=(8.9,6),layout='constrained')
ax[0].plot(t,x,label='x=Re(z)');ax[0].plot(t,z.imag,'--',label='H{x}=Im(z)');ax[0].plot(t,abs(z),':',label='|z|')
ax[0].set(xlim=(0,.25),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Dwa rzuty i jeden moduł');ax[0].legend()
for label,v,ls in [('x',x,'-'),('z',z,'--')]:
    f,F=centered_spectrum(v,fs);ax[1].plot(f,abs(F),ls,label=label)
ax[1].set(xlim=(-40,40),xlabel='Częstotliwość [Hz]',ylabel='|DFT| / N [j.u.]',title='b) Jednostronność przez anulowanie jednej półosi');ax[1].legend()
RESULTS['E6.2']={'max_manual_SciPy_error':max(errors),'pure_tone_energy_ratio':float(np.sum(abs(z)**2)/np.sum(x*x))}
save_figure(fig,FIG,'fig_06_02_analytic_signal')


OK: E6.2: Hilbert cos = sin dla całkowitej liczby okresów
OK: E6.2: ręczna maska = SciPy, N=499
OK: E6.2: Re(z)=x, N=499
OK: E6.2: zwykła ujemna półoś usunięta, N=499
OK: E6.2: bilans energii z DC/Nyquistą, N=499
OK: E6.2: ręczna maska = SciPy, N=500
OK: E6.2: Re(z)=x, N=500
OK: E6.2: zwykła ujemna półoś usunięta, N=500
OK: E6.2: bilans energii z DC/Nyquistą, N=500
Figura: figures/generated/fig_06_02_analytic_signal.png (+ SVG)


## E6.3 — test Re(z)=x nie wykrywa brzegów; rys. 6.3

Cosinus17 Hz, fs500 Hz, fragment około6.35 okresu (N z round).
Odniesienie kwadraturowe sin znane analitycznie. Błąd na całym fragmencie
i po odrzuceniu10% próbek z obu stron; nie jest to uniwersalny margines.
Dodatkowo paczki Gaussa: fc42 Hz, sigma0.035 s, centra0.50 i0.055 s, T1 s.

![Rys. 6.3](../figures/generated/fig_06_03_edges.png)


In [4]:
fs=500.;N=round(6.35/17*fs);t=np.arange(N)/fs
x=np.cos(2*np.pi*17*t);z=signal.hilbert(x);truth=np.sin(2*np.pi*17*t)
g=round(.1*N);inner=np.zeros(N,bool);inner[g:-g]=True
check('E6.3: część rzeczywista nadal dokładna',np.max(abs(z.real-x))<1e-12)
whole_error=rmse(z.imag,truth);inner_error=rmse(z.imag[inner],truth[inner])
check('E6.3: błąd H pozostaje mimo Re(z)=x',whole_error>.01)
fig,ax=plt.subplots(2,1,figsize=(8.8,6),layout='constrained')
ax[0].plot(t,z.imag-truth,label='Im(hilbert(x)) − sin')
ax[0].axvspan(0,t[g],color='0.7',alpha=.2);ax[0].axvspan(t[-g],t[-1],color='0.7',alpha=.2)
ax[0].set(xlabel='Czas [s]',ylabel='Błąd [j.u.]',title='a) Nieokresowy wycinek i nieznane otoczenie');ax[0].legend()
tt=np.arange(500)/fs;packet_rows=[]
for center,ls in [(.5,'-'),(.055,'--')]:
    c,A=packet(tt,center,.035,42);est=abs(signal.hilbert(c.real))
    ax[1].plot(tt,est-A,ls,label=f'Centrum {center:g} s')
    packet_rows.append({'center_s':center,'envelope_RMSE':rmse(est,A)})
ax[1].set(xlabel='Czas [s]',ylabel='Błąd obwiedni [j.u.]',title='b) Paczka ucięta przez granicę ma inny błąd');ax[1].legend()
RESULTS['E6.3']={'quadrature_RMSE_full':whole_error,'quadrature_RMSE_inner':inner_error,'packets':packet_rows}
save_figure(fig,FIG,'fig_06_03_edges')


OK: E6.3: część rzeczywista nadal dokładna
OK: E6.3: błąd H pozostaje mimo Re(z)=x
Figura: figures/generated/fig_06_03_edges.png (+ SVG)


## E6.4 — obwiednia, prostownik i interferencja; rys. 6.4

fs500 Hz, T10 s, A=1.2+0.55*sin(2π0.8t), nośne45 Hz oraz0.5 Hz.
Prostowanie: LP Butterworth4,3 Hz, SOS przód–tył odd/padlen15, korektaπ/2.
Dudnienia: cos42 Hz + cos45 Hz, amplitudy po1 j.u.; wszystkie tony na siatce DFT.
Pełne rozdzielenie pasm modelu jest warunkiem wystarczającym dla poprawnego AM,
a Gauss z poprzedniej sekcji ma jedynie przybliżone ograniczenie pasma.

![Rys. 6.4](../figures/generated/fig_06_04_envelope.png)


In [5]:
fs=500.;t=np.arange(5000)/fs;A=1.2+.55*np.sin(2*np.pi*.8*t)
x=A*np.cos(2*np.pi*45*t);env=abs(signal.hilbert(x));rect=abs(x)
sos=signal.butter(4,3,fs=fs,output='sos')
rectlp=np.pi/2*signal.sosfiltfilt(sos,rect,padtype='odd',padlen=15)
bad=A*np.cos(2*np.pi*.5*t);badenv=abs(signal.hilbert(bad))
beats=np.cos(2*np.pi*42*t)+np.cos(2*np.pi*45*t)
beat_env=abs(signal.hilbert(beats));beat_truth=2*abs(np.cos(np.pi*3*t))
check('E6.4: AM45 Hz odtwarza A',rmse(env,A)<1e-11)
check('E6.4: brak rozdzielenia psuje interpretację A',rmse(badenv,A)>.1)
check('E6.4: obwiednia sumy pokazuje dudnienia',rmse(beat_env,beat_truth)<1e-11)
fig,ax=plt.subplots(3,1,figsize=(9,7.4),layout='constrained')
for name,y,ls in [('A odniesienia',A,'-'),('|x|',rect,':'),('|z|',env,'--'),('LP(|x|) × π/2',rectlp,'-.')]:
    ax[0].plot(t,y,ls,lw=.8 if name=='|x|' else 1.4,label=name)
ax[0].set(xlim=(1,3.6),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Wolna amplituda i nośna45 Hz');ax[0].legend(ncol=2)
ax[1].plot(t,A,label='A odniesienia');ax[1].plot(t,badenv,'--',label='|z|, nośna0,5 Hz')
ax[1].set(xlim=(0,5),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='b) Modulacja szybsza od nośnej');ax[1].legend()
ax[2].plot(t,beat_env,label='|hilbert(cos42+cos45)|');ax[2].plot(t,beat_truth,'--',label='2|cos(π·3t)|')
ax[2].set(xlim=(0,1.5),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='c) Dudnienia dwóch stałych amplitud');ax[2].legend()
mask=(t>=.6)&(t<9.4)
RESULTS['E6.4']={'Hilbert_AM_RMSE':rmse(env[mask],A[mask]),'rectified_LP_RMSE':rmse(rectlp[mask],A[mask]),
                  'raw_absolute_RMSE':rmse(rect[mask],A[mask]),'violated_AM_RMSE':rmse(badenv[mask],A[mask])}
save_figure(fig,FIG,'fig_06_04_envelope')


OK: E6.4: AM45 Hz odtwarza A
OK: E6.4: brak rozdzielenia psuje interpretację A
OK: E6.4: obwiednia sumy pokazuje dudnienia
Figura: figures/generated/fig_06_04_envelope.png (+ SVG)


## E6.5 — pochodna fazy, zanik amplitudy i pokrycie; rys. 6.5

Chirp fs500 Hz, T4 s, φ=2π(12t+0.5·14.5t²)+0.2, f_ref=12+14.5t Hz.
Zanik A=1−0.98*exp(−0.5*((t−2.1)/0.16)²), biały szum SD0.12 j.u., seed20260720.
np.gradient(unwrap(angle(z)),1/fs)/(2π), bez wygładzania; margines0.2 s.
Maska amplitudowa obejmuje też sąsiadów różnicy centralnej. Próg0.2*max|z|,
kontrola progów0…0.55. Pokrycie liczone względem próbek części wewnętrznej.

![Rys. 6.5](../figures/generated/fig_06_05_instantaneous_frequency.png)


In [6]:
fs=500.;t=np.arange(2000)/fs;phi=2*np.pi*(12*t+.5*14.5*t*t)+.2;fref=12+14.5*t
x=np.cos(phi);base=analytic_features(x,fs,guard_s=.2)
inner=(t>=.2)&(t<3.8)
base_mae=float(np.mean(abs(base['frequency_raw'][inner]-fref[inner])))
check('E6.5: chirp w środku bliski prawdzie',base_mae<.5)
check('E6.5: dwie równoważne konwersje gradientu do Hz',np.allclose(np.gradient(base['phase'])*fs/(2*np.pi),base['frequency_raw']))
rng=np.random.default_rng(20260720)
A=1-.98*np.exp(-.5*((t-2.1)/.16)**2)
noisy=A*x+rng.normal(0,.12,len(t))
feat=analytic_features(noisy,fs,.2,.2)
rows=[]
for threshold in np.linspace(0,.55,12):
    ft=analytic_features(noisy,fs,float(threshold),.2);valid=ft['valid']&inner
    rows.append({'threshold_fraction':float(threshold),'coverage_inner':float(valid.sum()/inner.sum()),
                 'MAE_Hz':float(np.mean(abs(ft['frequency_raw'][valid]-fref[valid]))) if valid.any() else None})
check('E6.5: odrzucone próbki są NaN',np.all(np.isnan(feat['frequency'][~feat['valid']])))
check('E6.5: podnoszenie progu nie zwiększa pokrycia',np.all(np.diff([r['coverage_inner'] for r in rows])<=1e-12))
fig,ax=plt.subplots(3,1,figsize=(9,7.7),layout='constrained')
ax[0].plot(t,feat['amplitude'],lw=.8,label='|z|');ax[0].plot(t,A,'--',label='A modelu');ax[0].axhline(feat['amplitude_threshold'],color='black',ls=':',label='Próg')
ax[0].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Zanik zwiększa względny wpływ szumu');ax[0].legend(ncol=3)
ax[1].plot(t,feat['frequency_raw'],color='0.75',lw=.7,label='Wynik surowy');ax[1].plot(t,feat['frequency'],lw=.9,label='Próbki zaakceptowane');ax[1].plot(t,fref,'k--',lw=1.6,label='Odniesienie')
ax[1].set(ylim=(-60,150),xlabel='Czas [s]',ylabel='Częstotliwość [Hz]',title='b) Brak estymaty nie jest częstotliwością0 Hz');ax[1].legend(ncol=3)
ax[2].plot([r['coverage_inner'] for r in rows],[r['MAE_Hz'] for r in rows],'o-')
ax[2].set(xlabel='Pokrycie części wewnętrznej [1]',ylabel='MAE [Hz]',title='c) Zmiana progu wybiera inny zbiór próbek')
RESULTS['E6.5']={'clean_chirp_MAE_inner_Hz':base_mae,'threshold_sweep':rows,
                 'selected_coverage_inner':float((feat['valid']&inner).sum()/inner.sum())}
save_figure(fig,FIG,'fig_06_05_instantaneous_frequency')


OK: E6.5: chirp w środku bliski prawdzie
OK: E6.5: dwie równoważne konwersje gradientu do Hz
OK: E6.5: odrzucone próbki są NaN
OK: E6.5: podnoszenie progu nie zwiększa pokrycia
Figura: figures/generated/fig_06_05_instantaneous_frequency.png (+ SVG)


## E6.6 — separacja półosi i kombinacje Hilberta; rys. 6.6

fs500 Hz, T1.2 s. Paczka+: centrum0.48 s, σ0.095 s, f62 Hz, A1.
Paczka−: centrum0.66 s, σ0.110 s, f47 Hz, A0.8, faza0.4 rad.
Brak szumu; znamy obie składowe. DC/Nyquista wydzielone jako reszta, bez mnożnika2.
Q±H(D): porównanie po usunięciu tych samych szczególnych prążków z D/Q;
0.5*|hilbert(Q+H(D))| jest obwiednią dodatniego wyjścia w tej konwencji.

![Rys. 6.6](../figures/generated/fig_06_06_direction_separation.png)


In [7]:
fs=500.;t=np.arange(600)/fs
positive,Ap=packet(t,.48,.095,62,1,1.)
negative,An=packet(t,.66,.110,47,-1,.8,.4)
c=positive+negative
sp,sn,residual=split_directions(c,fs)
check('E6.6: rozdział plus reszta odtwarza wejście',np.allclose(sp+sn+residual,c,atol=1e-12))
errp=float(np.linalg.norm(sp-positive)/np.linalg.norm(positive));errn=float(np.linalg.norm(sn-negative)/np.linalg.norm(negative))
check('E6.6: mały błąd znanych paczek oddalonych od zera',errp<1e-3 and errn<1e-3)
cleaned=sp+sn;D,Q=cleaned.real,cleaned.imag;HD=signal.hilbert(D).imag
yp=Q+HD;yn=Q-HD
hilp=.5*abs(signal.hilbert(yp));hiln=.5*abs(signal.hilbert(yn))
check('E6.6: kombinacja Hilberta zgodna z dodatnią projekcją',np.allclose(hilp,abs(sp),atol=1e-12))
check('E6.6: kombinacja Hilberta zgodna z ujemną projekcją',np.allclose(hiln,abs(sn),atol=1e-12))
flip_p,flip_n,flip_r=split_directions(c.conj(),fs)
check('E6.6: zmiana znaku Q zamienia półosie',np.allclose(flip_p,sn.conj(),atol=1e-12) and np.allclose(flip_n,sp.conj(),atol=1e-12))
fig,ax=plt.subplots(3,1,figsize=(9,7.4),layout='constrained')
f,C=centered_spectrum(c,fs);ax[0].plot(f,abs(C))
ax[0].set(xlim=(-100,100),xlabel='Częstotliwość [Hz]',ylabel='|DFT| / N [j.u.]',title='a) Dwie półosie mierzonego sygnału D+jQ')
for a,est,ref,hil,label in [(ax[1],abs(sp),Ap,hilp,'dodatnia'),(ax[2],abs(sn),An,hiln,'ujemna')]:
    a.plot(t,ref,'k',label='Obwiednia odniesienia');a.plot(t,est,'--',label='Maska i IFFT');a.plot(t,hil,':',label='Kombinacja Hilberta /2')
    a.set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title=f'Półoś {label}');a.legend(ncol=3,fontsize=7)
RESULTS['E6.6']={'relative_error_positive':errp,'relative_error_negative':errn,
    'Hilbert_positive_max_abs_difference':float(np.max(abs(hilp-abs(sp)))),
    'special_bins_residual_norm':float(np.linalg.norm(residual))}
save_figure(fig,FIG,'fig_06_06_direction_separation')


OK: E6.6: rozdział plus reszta odtwarza wejście
OK: E6.6: mały błąd znanych paczek oddalonych od zera
OK: E6.6: kombinacja Hilberta zgodna z dodatnią projekcją
OK: E6.6: kombinacja Hilberta zgodna z ujemną projekcją
OK: E6.6: zmiana znaku Q zamienia półosie
Figura: figures/generated/fig_06_06_direction_separation.png (+ SVG)


## E6.7 — niedoskonała kwadratura tworzy obraz; rys. 6.7

Jedyny fizycznie zadany w modelu składnik exp(j2π62t), fs500 Hz, T2 s,
całkowita liczba okresów. D=cos+0.12, Q=0.72*sin(φ+13°)−0.08.
Przed separacją odjęto średnie kanałów. Przeciek = energia wyjścia ujemnego/
energia dodatniego dla całego kontrolnego tonu. Nie używamy nakładających się
paczek jako rzekomo czystego fragmentu kalibracyjnego.

![Rys. 6.7](../figures/generated/fig_06_07_iq_imbalance.png)


In [8]:
fs=500.;t=np.arange(1000)/fs;phi=2*np.pi*62*t
ideal=np.exp(1j*phi);D=np.cos(phi)+.12;Q=.72*np.sin(phi+np.deg2rad(13))-.08
imperfect=(D-D.mean())+1j*(Q-Q.mean())
ip,inn,_=split_directions(ideal,fs);bp,bn,_=split_directions(imperfect,fs)
leak_ideal=float(np.sum(abs(inn)**2)/np.sum(abs(ip)**2))
leak_bad=float(np.sum(abs(bn)**2)/np.sum(abs(bp)**2))
check('E6.7: obraz powstaje mimo usunięcia offsetu',leak_bad>.01 and leak_ideal<1e-20)
fig,ax=plt.subplots(1,2,figsize=(9.4,4),layout='constrained')
ax[0].plot(ideal[:30].real,ideal[:30].imag,label='Idealna para');ax[0].plot(imperfect[:30].real,imperfect[:30].imag,'--',label='Nierówność i błąd fazy')
ax[0].set(xlabel='D [j.u.]',ylabel='Q [j.u.]',aspect='equal',title='a) Od koła do elipsy');ax[0].legend(fontsize=7)
for value,name,ls in [(ideal,'Idealna','-'),(imperfect,'Niedoskonała','--')]:
    f,C=centered_spectrum(value,fs);ax[1].plot(f,abs(C),ls,label=name)
ax[1].set(xlim=(-90,90),xlabel='Częstotliwość [Hz]',ylabel='|DFT| / N [j.u.]',title='b) Drugi prążek bez drugiego kierunku w modelu');ax[1].legend()
RESULTS['E6.7']={'ideal_image_energy_ratio':leak_ideal,'imperfect_image_energy_ratio':leak_bad,
                 'imperfect_image_dB':float(10*np.log10(leak_bad))}
save_figure(fig,FIG,'fig_06_07_iq_imbalance')


OK: E6.7: obraz powstaje mimo usunięcia offsetu
Figura: figures/generated/fig_06_07_iq_imbalance.png (+ SVG)


## E6.8 — ACF, SAD i SSD w tym samym zakresie; rys. 6.8

fs100 Hz, T20 s, nominalny rytm142/min; Gaussy σ0.055 s od0.4 s, szumSD0.08,
seed20260721. Przygotowanie: detrend liniowy, SD dzielnikN. Lagi30…75.
ACF: średnia iloczynów/(globalny drugi moment), SAD/SSD: średnie różnice
bezwzględne/kwadratowe, wszystkie z dzielnikiemN−k. Nie zakładamy |ACF|<=1.
Wybór globalnego optimum; do raportowania potrzebny jest też lokalny szczyt
funkcji score, po odwróceniu SAD/SSD i normalizacji zakresu do0…1.

![Rys. 6.8](../figures/generated/fig_06_08_period_objectives.png)


In [9]:
fs=100.;t=np.arange(2000)/fs;ref_rate=142.;T=60/ref_rate
x=np.zeros(len(t))
for at in np.arange(.4,20,T):x+=np.exp(-.5*((t-at)/.055)**2)
x+=np.random.default_rng(20260721).normal(0,.08,len(x))
low,high=lag_bounds(fs);lags=np.arange(low,high+1);obj=objectives(x,lags)
check('E6.8: zakres przy100 Hz to30…75',low==30 and high==75)
for fss in [50.,73.,200.]:
    lo,hi=lag_bounds(fss);rates=60*fss/np.arange(lo,hi+1)
    check(f'E6.8: ceil/floor zachowują wszystkie kandydaty w80–200, fs={fss}',rates.min()>=80-1e-12 and rates.max()<=200+1e-12)
fig,ax=plt.subplots(3,1,figsize=(8.9,7.2),sharex=True,layout='constrained');rows=[]
for a,method in zip(ax,['ACF','SAD','SSD']):
    result=select_optimum(obj[method],lags,method);lag=result['lag'];rate=60*fs/lag
    a.plot(lags/fs,obj[method]);a.scatter([lag/fs],[obj[method][np.where(lags==lag)[0][0]]],color='#B25324',zorder=3)
    a.axvline(T,color='black',ls=':',label='Okres odniesienia')
    a.set(ylabel=method+(' [1]' if method=='ACF' else ' [po normalizacji]'),title=f'{method}: k={lag}, r={rate:.2f}/min')
    a.legend(fontsize=7)
    check(f'E6.8: {method} blisko142/min na siatce lagów',abs(rate-ref_rate)<3)
    rows.append({'method':method,'lag':lag,'period_s':lag/fs,'rate_per_min':rate,'quality':result['quality'],'gap':result['gap']})
ax[-1].set_xlabel('Opóźnienie [s]')
RESULTS['E6.8']=rows
save_figure(fig,FIG,'fig_06_08_period_objectives')


OK: E6.8: zakres przy100 Hz to30…75
OK: E6.8: ceil/floor zachowują wszystkie kandydaty w80–200, fs=50.0
OK: E6.8: ceil/floor zachowują wszystkie kandydaty w80–200, fs=73.0
OK: E6.8: ceil/floor zachowują wszystkie kandydaty w80–200, fs=200.0
OK: E6.8: ACF blisko142/min na siatce lagów
OK: E6.8: SAD blisko142/min na siatce lagów
OK: E6.8: SSD blisko142/min na siatce lagów
Figura: figures/generated/fig_06_08_period_objectives.png (+ SVG)


## E6.9 — harmoniczna może mieścić się w zakresie; rys. 6.9

fs120 Hz, T20 s. ModelA: cykl90/min ma dwie identyczne paczkiσ0.045 s,
odstępT/2; sygnał obserwowany ma też rzeczywistą powtarzalnośćT/2.
ModelB: cykle180/min o naprzemiennych amplitudach1/0.5 i szerokościach0.04/0.085 s.
W obu tych modelach T/2 lub2T mieści się w tym samym zakresie80–200/min.
Zakres jest stały. Przy fs120 Hz okresy1/3 i2/3 s leżą dokładnie na siatce lagów;
wybór usuwa z tej próby konkurencję błędów kwantyzacji. Brak szumu.

![Rys. 6.9](../figures/generated/fig_06_09_harmonic_errors.png)


In [10]:
fs=120.;t=np.arange(2400)/fs;half=np.zeros(len(t));double=np.zeros(len(t))
for at in np.arange(.4,20,60/90):
    half+=np.exp(-.5*((t-at)/.045)**2)+np.exp(-.5*((t-at-60/90/2)/.045)**2)
for i,at in enumerate(np.arange(.4,20,60/180)):
    sigma,amp=(.04,1.) if i%2==0 else (.085,.5)
    double+=amp*np.exp(-.5*((t-at)/sigma)**2)
low,high=lag_bounds(fs);lags=np.arange(low,high+1)
fig,ax=plt.subplots(2,2,figsize=(9.5,6.2),layout='constrained');all_rows=[]
for row,(label,x,rate_ref) in enumerate([('Dwie paczki w cyklu',half,90.),('Naprzemienna morfologia',double,180.)]):
    obj=objectives(x,lags);rows=[]
    ax[row,0].plot(t,x);ax[row,0].set(xlim=(.3,2.5),xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title=label)
    for method,ls in [('ACF','-'),('SAD','--'),('SSD','-.')]:
        result=select_optimum(obj[method],lags,method);rate=60*fs/result['lag']
        rows.append({'method':method,'rate_per_min':rate,'lag':result['lag'],'reference_cycle_rate':rate_ref})
        ax[row,1].plot(lags/fs,result['score'],ls,label=f'{method}: {rate:.1f}/min')
    ax[row,1].axvline(60/rate_ref,color='black',ls=':',label='Cykl zadany w modelu')
    ax[row,1].set(xlabel='Opóźnienie [s]',ylabel='Score [1]',title='Zgodność metod nie identyfikuje cyklu fizjologicznego');ax[row,1].legend(fontsize=7)
    all_rows.append({'case':label,'estimates':rows})
check('E6.9: podwojona częstość w modelu dwóch paczek',all_rows[0]['estimates'][0]['rate_per_min']>170)
check('E6.9: obniżony rytm przy naprzemiennej morfologii',all_rows[1]['estimates'][0]['rate_per_min']<100)
gridfs=100.;gridt=np.arange(2000)/gridfs;gridx=np.zeros(len(gridt))
for at in np.arange(.4,20,60/90):gridx+=np.exp(-.5*((gridt-at)/.045)**2)+np.exp(-.5*((gridt-at-60/90/2)/.045)**2)
glo,ghi=lag_bounds(gridfs);glags=np.arange(glo,ghi+1);gobj=objectives(gridx,glags)
grid_rates={m:float(60*gridfs/select_optimum(gobj[m],glags,m)['lag']) for m in gobj}
RESULTS['E6.9']={'fs_Hz':fs,'cases':all_rows,'two_identical_packets_at100Hz_per_min':grid_rates}
save_figure(fig,FIG,'fig_06_09_harmonic_errors')


OK: E6.9: podwojona częstość w modelu dwóch paczek
OK: E6.9: obniżony rytm przy naprzemiennej morfologii
Figura: figures/generated/fig_06_09_harmonic_errors.png (+ SVG)


## E6.10 — od D/Q do lokalnego rytmu; rys. 6.10

fs_raw500 Hz, T32 s, seed20260719. Profil r(t)=120+40*sigmoid((t−12)/3)
−12*sigmoid((t−24)/2.5). Zdarzenia przez całkę r/60 i kolejne pełne cykle,
pierwsze0.5 s. Paczki+: σ0.030 s, f68+3sin(0.3i); paczki−: opóźnione0.145 s,
σ0.042 s, f52+2cos(0.4i), amplituda0.75 dodatniej.
Zanik14–17 s: mnożnik0.03; artefakt23–23.35 s: zespolony szumSD2.5;
szum kanałówSD0.06. To generator kontrolowany, nie model wszystkich ruchów płodu.

Półosie→|c+|+|c−|→LP8 Hz Butterworth4 przód–tył odd/padlen15→resample_poly1/5,
Kaiser5, constant. fs_env100 Hz; nowe lagi30…75. Okna1.5/3/5 s, krok0.25 s,
środek próbek okna jako czas. Próg prominencji0.10; aktywność95.percentyl−5.percentyl,
próg0.42 rozstępu z jawnego aktywnego fragmentu2–10 s. Nie używamy prawdy do akceptacji.
Referencja oceny: nominalny profil w środku, okno kwalifikuje się przy>=80% próbek
poza zanikiem14–17 s. Artefakt pozostaje w ocenie; brzegi oceniane tym samym modelem.

![Rys. 6.10](../figures/generated/fig_06_10_rhythm_track.png)


In [11]:
traw,c,ev,drop,artifact=moving_dq()
cp,cn,residual=split_directions(c,500.)
activity_raw=abs(cp)+abs(cn)
sos=signal.butter(4,8,fs=500.,output='sos')
smoothed=signal.sosfiltfilt(sos,activity_raw,padtype='odd',padlen=15)
env=signal.resample_poly(smoothed,1,5,window=('kaiser',5.),padtype='constant')
fsenv=100.;te=np.arange(len(env))/fsenv;ref=rate_profile(te);usable=~((te>=14)&(te<=17))
active=(te>2)&(te<10)
activity_threshold=.42*(np.percentile(env[active],95)-np.percentile(env[active],5))
tracks={};track_rows=[]
for window in [1.5,3.,5.]:
    track=estimate_track(env,fsenv,window,.25,.1,activity_threshold)
    met,rref,elig,frac=evaluate_track(track,te,ref,usable,.8)
    tracks[window]=(track,met,rref,elig,frac);track_rows.append({'window_s':window,**met})
    check(f'E6.10: liczone środki próbek i stały krok, W={window}',np.isclose(track['time_s'][0],(round(window*fsenv)-1)/(2*fsenv)) and np.allclose(np.diff(track['time_s']),.25))
    check(f'E6.10: maska i rytmy zachowują zakres, W={window}',np.all((track['rate'][np.isfinite(track['rate'])]>=80)&(track['rate'][np.isfinite(track['rate'])]<=200)))
fig,ax=plt.subplots(3,1,figsize=(9.4,8),layout='constrained')
ax[0].plot(traw,c.real,lw=.5,label='D');ax[0].plot(traw,c.imag,lw=.5,alpha=.6,label='Q')
ax[0].set(xlabel='Czas [s]',ylabel='Amplituda [j.u.]',title='a) Paczki kierunkowe, zanik i artefakt');ax[0].legend(ncol=2)
ax[1].plot(te,env,lw=.9);ax[1].set(xlabel='Czas [s]',ylabel='Aktywność [j.u.]',title='b) Suma modułów kierunków po filtracji i resamplingu')
ax[2].plot(te,np.where(usable,ref,np.nan),'k',lw=1.8,label='Profil odniesienia')
for w,ls in [(1.5,'-'),(3.,'--'),(5.,'-.')]:
    track=tracks[w][0];ax[2].plot(track['time_s'],track['rate'],ls,marker='.',ms=3,lw=.9,label=f'Okno {w:g} s')
ax[2].set(ylim=(75,205),xlabel='Czas środka okna [s]',ylabel='Rytm [1/min]',title='c) Długość okna zmienia informację, nie tylko gęstość punktów');ax[2].legend(ncol=2)
for a in ax:a.axvspan(14,17,color='0.7',alpha=.2)
RESULTS['E6.10']={'activity_threshold':float(activity_threshold),'window_comparison':track_rows,'events':len(ev),
                  'fs_raw_Hz':500.,'fs_envelope_Hz':100.,'N_output':len(env)}
save_figure(fig,FIG,'fig_06_10_rhythm_track')


OK: E6.10: liczone środki próbek i stały krok, W=1.5
OK: E6.10: maska i rytmy zachowują zakres, W=1.5
OK: E6.10: liczone środki próbek i stały krok, W=3.0
OK: E6.10: maska i rytmy zachowują zakres, W=3.0
OK: E6.10: liczone środki próbek i stały krok, W=5.0
OK: E6.10: maska i rytmy zachowują zakres, W=5.0
Figura: figures/generated/fig_06_10_rhythm_track.png (+ SVG)


## E6.11 — jakość, mapa czas–lag i odmowa wyniku; rys. 6.11

Ten sam sygnał z E6.10, okno3 s, krok0.25 s. Próg jakości0…0.90;
próg aktywności pozostaje stały. Score=(ACF−min)/(max−min) w zakresie lagów,
jakość=prominencja wybranego szczytu score (brzeg nie jest lokalnym szczytem).
Pokrycie raportowane oddzielnie dla wszystkich i kwalifikowanych okien;
fałszywa akceptacja dotyczy okien niespełniających80% próbek poza zanikiem.
Reguła szybkości18(1/min)/s tylko odrzuca, bez podtrzymywania poprzedniej wartości.

![Rys. 6.11](../figures/generated/fig_06_11_quality_lag_map.png)


In [12]:
selected,selected_metrics,rref,eligible,fraction=tracks[3.]
rows=[]
for threshold in np.linspace(0,.9,13):
    track=estimate_track(env,fsenv,3.,.25,float(threshold),activity_threshold)
    met,_,_,_=evaluate_track(track,te,ref,usable,.8)
    rows.append({'threshold':float(threshold),**met})
check('E6.11: wyższy próg nie zwiększa liczby akceptacji',np.all(np.diff([r['accepted_eligible_count']+r['false_accept_count'] for r in rows])<=0))
rate_filtered,rejected=reject_fast_changes(selected['rate'],selected['time_s'],18.)
check('E6.11: reguła szybkości nie wypełnia braków',np.all(np.isnan(rate_filtered[np.isnan(selected['rate'])])))
check('E6.11: odrzucone skoki pozostają NaN',np.all(np.isnan(rate_filtered[rejected])))
check('E6.11: stały sygnał nie daje sztucznego okresu',np.all(np.isnan(estimate_track(np.ones(500),100,3.,.25,0,0)['rate'])))
test_edge=select_optimum(np.arange(10.),np.arange(30,40))
check('E6.11: optimum na granicy nie jest akceptowanym szczytem',not test_edge['interior_peak'])
fig,ax=plt.subplots(3,1,figsize=(9.4,8.3),layout='constrained')
im=ax[0].pcolormesh(selected['time_s'],selected['lags']/fsenv,selected['score_map'].T,shading='auto',cmap='viridis',vmin=0,vmax=1)
ax[0].plot(selected['time_s'],selected['lag']/fsenv,'w.',ms=3,label='Zaakceptowane opóźnienie')
ax[0].set(xlabel='Czas środka okna [s]',ylabel='Opóźnienie [s]',title='a) Mapa funkcji celu, nie spektrogram');ax[0].legend(fontsize=7)
fig.colorbar(im,ax=ax[0],label='Score [1]')
ax[1].plot(selected['time_s'],selected['quality'],label='Prominencja score');ax[1].axhline(.1,color='black',ls=':',label='Próg0,10')
right=ax[1].twinx();right.grid(False)
right.plot(selected['time_s'],selected['activity']/max(activity_threshold,1e-12),'--',color='#B25324',label='Aktywność / próg')
right.axhline(1.,color='#B25324',ls=':',alpha=.7);right.set_ylabel('Aktywność / próg [1]',color='#B25324')
ax[1].set(xlabel='Czas [s]',ylabel='Prominencja [1]',ylim=(0,1.08),title='b) Wyrazistość i aktywność mierzą różne własności')
handles,labels=ax[1].get_legend_handles_labels();h2,l2=right.get_legend_handles_labels();ax[1].legend(handles+h2,labels+l2,ncol=3,fontsize=7)
coverage=[r['coverage_eligible_windows'] for r in rows];maes=[r['MAE_per_min'] for r in rows]
ax[2].plot(coverage,maes,'o-',label='Zmiana progu jakości')
ax[2].set(xlabel='Pokrycie kwalifikowanych okien [1]',ylabel='MAE [1/min]',title='c) Mały błąd może dotyczyć wybranego podzbioru');ax[2].legend()
RESULTS['E6.11']={'quality_sweep':rows,'selected_metrics':selected_metrics,'rejected_change_count':int(rejected.sum())}
table='| Okno [s] | MAE [1/min] | RMSE [1/min] | Pokrycie kwalifikowanych | Fałszywe akceptacje / okna niekwalifikowane |\n|---:|---:|---:|---:|---:|\n'
for r in RESULTS['E6.10']['window_comparison']:
    values=[f"{r['window_s']:.1f}",f"{r['MAE_per_min']:.2f}",f"{r['RMSE_per_min']:.2f}",f"{100*r['coverage_eligible_windows']:.1f}%",f"{r['false_accept_count']}/{r['ineligible_count']}"]
    table+='| '+' | '.join(v.replace('.',',') for v in values)+' |\n'
(ROOT/'data/ch06_track_table.md').write_text(table, encoding='utf-8', newline='\n')
save_figure(fig,FIG,'fig_06_11_quality_lag_map')


OK: E6.11: wyższy próg nie zwiększa liczby akceptacji
OK: E6.11: reguła szybkości nie wypełnia braków
OK: E6.11: odrzucone skoki pozostają NaN
OK: E6.11: stały sygnał nie daje sztucznego okresu
OK: E6.11: optimum na granicy nie jest akceptowanym szczytem
Figura: figures/generated/fig_06_11_quality_lag_map.png (+ SVG)


## E6.12 — jedna faza nie rozdziela dwóch torów; rys. 6.12

fs400 Hz, T5 s. Dwie znane fazy z częstotliwościami18+8.4t i72−8.4t Hz,
amplitudy1 i0.75. x=cosφ1+0.75cosφ2. Sygnał analityczny i gradient jego fazy,
bez przypisywania wyniku żadnej ze składowych. Brak STFT w tym pakiecie;
porównanie uzasadnia dopiero następny rozdział.

![Rys. 6.12](../figures/generated/fig_06_12_multicomponent_limit.png)


In [13]:
fs=400.;t=np.arange(2000)/fs
f1=18+8.4*t;f2=72-8.4*t
phi1=2*np.pi*(18*t+.5*8.4*t*t);phi2=2*np.pi*(72*t-.5*8.4*t*t)+.3
x=np.cos(phi1)+.75*np.cos(phi2)
feat=analytic_features(x,fs,0,.2)
fig,ax=plt.subplots(2,1,figsize=(9,6),sharex=True,layout='constrained')
ax[0].plot(t,feat['amplitude'],lw=.8);ax[0].set(ylabel='|z| [j.u.]',title='a) Interferencja zmienia moduł sumy')
ax[1].plot(t,feat['frequency'],lw=.7,label='Częstotliwość fazy sumy');ax[1].plot(t,f1,'k--',lw=1.6,label='Składowa1');ax[1].plot(t,f2,'-.',lw=1.6,label='Składowa2')
ax[1].set(xlabel='Czas [s]',ylabel='Częstotliwość [Hz]',ylim=(-80,180),title='b) Jedna krzywa nie jest dwiema lokalnymi częstotliwościami');ax[1].legend(ncol=3)
inner=feat['valid'];dist=np.minimum(abs(feat['frequency_raw'][inner]-f1[inner]),abs(feat['frequency_raw'][inner]-f2[inner]))
check('E6.12: faza sumy nie odtwarza obu torów',np.mean(dist)>1)
RESULTS['E6.12']={'mean_distance_to_nearest_true_component_Hz':float(np.mean(dist)),
                  'fraction_more_than5Hz_from_both':float(np.mean(dist>5))}
save_figure(fig,FIG,'fig_06_12_multicomponent_limit')


OK: E6.12: faza sumy nie odtwarza obu torów
Figura: figures/generated/fig_06_12_multicomponent_limit.png (+ SVG)


## Raport wykonania

Wersje i parametry zapisano obok rozdziału; `data/ch06_results.json` zawiera
liczby z eksperymentów. Odrzucone estymaty w kodzie i na wykresach pozostają NaN;
JSON przechowuje metryki, a nie sztucznie uzupełnione ślady.
Kontrole dotyczą modeli i obliczeń, nie skuteczności klinicznej.


In [14]:
for i in range(1,13):
    for ext in ['png','svg']:
        check(f'Figura6.{i} {ext}',len(list(FIG.glob(f'fig_06_{i:02d}_*.{ext}')))==1)
RESULTS['audit']={'passed_checks':len(CHECKS),'checks':CHECKS,'experiments':12,'figures':12,'external_data':False}
(ROOT/'data/ch06_results.json').write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2,allow_nan=False)+'\n', encoding='utf-8', newline='\n')
(ROOT/'chapter_notes/ch06_environment.json').write_text(json.dumps(ENV,ensure_ascii=False,indent=2)+'\n', encoding='utf-8', newline='\n')
print('Zakończono; liczba kontroli:',len(CHECKS))


OK: Figura6.1 png
OK: Figura6.1 svg
OK: Figura6.2 png
OK: Figura6.2 svg
OK: Figura6.3 png
OK: Figura6.3 svg
OK: Figura6.4 png
OK: Figura6.4 svg
OK: Figura6.5 png
OK: Figura6.5 svg
OK: Figura6.6 png
OK: Figura6.6 svg
OK: Figura6.7 png
OK: Figura6.7 svg
OK: Figura6.8 png
OK: Figura6.8 svg
OK: Figura6.9 png
OK: Figura6.9 svg
OK: Figura6.10 png
OK: Figura6.10 svg
OK: Figura6.11 png
OK: Figura6.11 svg
OK: Figura6.12 png
OK: Figura6.12 svg
Zakończono; liczba kontroli: 72
